In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc qiskit-ibm-catalog
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# Funzione IBM Circuit

> **Note:** * Le Qiskit Functions sono una funzionalità sperimentale disponibile solo per gli utenti di IBM Quantum&reg; Premium Plan, Flex Plan e On-Prem (tramite IBM Quantum Platform API) Plan. Sono in stato di anteprima e soggette a modifiche.

## Panoramica
IBM&reg; Circuit function accetta come input [PUB astratti](./primitive-input-output) e restituisce come output valori di aspettazione mitigati. Questa circuit function include una pipeline automatizzata e personalizzata per consentire ai ricercatori di concentrarsi sulla scoperta di algoritmi e applicazioni.

## Descrizione
Dopo aver inviato il tuo PUB, i tuoi circuiti astratti e le osservabili vengono automaticamente transpilati, eseguiti sull'hardware e post-elaborati per restituire valori di aspettazione mitigati. Per farlo, vengono combinati i seguenti strumenti:

- [Qiskit Transpiler Service](./qiskit-transpiler-service), che include la selezione automatica di pass di transpilazione guidati dall'intelligenza artificiale ed euristici per tradurre i tuoi circuiti astratti in circuiti ISA ottimizzati per l'hardware
- [Soppressione e mitigazione degli errori necessarie per il calcolo a scala utilitaristica](./error-mitigation-and-suppression-techniques), tra cui measurement twirling e gate twirling, dynamical decoupling, Twirled Readout Error eXtinction (TREX), Zero-Noise Extrapolation (ZNE) e Probabilistic Error Amplification (PEA)
- [Qiskit Runtime Estimator](./get-started-with-primitives), per eseguire ISA PUB sull'hardware e restituire valori di aspettazione mitigati

![IBM Circuit function](../docs/images/guides/ibm-circuit-function/ibm-circuit-function.svg)

## Inizia
Autenticati usando la tua [chiave API](http://quantum.cloud.ibm.com/) e seleziona la Qiskit Function come segue. (Questo snippet presuppone che tu abbia già [salvato il tuo account](/guides/functions#install-qiskit-functions-catalog-client) nel tuo ambiente locale.)

In [1]:
from qiskit_ibm_catalog import QiskitFunctionsCatalog

catalog = QiskitFunctionsCatalog(channel="ibm_quantum_platform")

function = catalog.load("ibm/circuit-function")

## Esempio
Per iniziare, prova questo esempio di base:

In [2]:
from qiskit.circuit.random import random_circuit
from qiskit_ibm_runtime import QiskitRuntimeService

# You can skip this step if you have a target backend, e.g.
# backend_name = "ibm_brisbane"
# You'll need to specify the credentials when initializing QiskitRuntimeService, if they were not previously saved.
service = QiskitRuntimeService()
backend = service.least_busy(operational=True, simulator=False)

circuit = random_circuit(num_qubits=2, depth=2, seed=42)
observable = "Z" * circuit.num_qubits
pubs = [(circuit, observable)]

job = function.run(
    # Use `backend_name=backend_name` if you didn't initialize a backend object
    backend_name=backend.name,
    pubs=pubs,
)

Controlla lo [stato](/guides/functions#check-job-status) del tuo workload Qiskit Function o recupera i [risultati](/guides/functions#retrieve-results) come segue:

In [3]:
print(job.status())
result = job.result()

QUEUED


The results have the same format as an [Estimator result](/docs/guides/primitive-input-output#estimator-output):

In [4]:
print(f"The result of the submitted job had {len(result)} PUB\n")
print(
    f"The associated PubResult of this job has the following DataBins:\n {result[0].data}\n"
)
print(f"And this DataBin has attributes: {result[0].data.keys()}")
print(
    f"The expectation values measured from this PUB are: \n{result[0].data.evs}"
)

The result of the submitted job had 1 PUB

The associated PubResult of this job has the following DataBins:
 DataBin(evs=np.ndarray(<shape=(), dtype=float64>), stds=np.ndarray(<shape=(), dtype=float64>), ensemble_standard_error=np.ndarray(<shape=(), dtype=float64>))

And this DataBin has attributes: dict_keys(['evs', 'stds', 'ensemble_standard_error'])
The expectation values measured from this PUB are: 
1.02116704805492


I risultati hanno lo stesso formato di un [risultato Estimator](/guides/primitive-input-output#estimator-output):

In [5]:
options = {"mitigation_level": 2}

job = function.run(backend_name=backend.name, pubs=pubs, options=options)

#### All available options

In addition to `mitigation_level`, the IBM Circuit function also provides a select number of advanced options that allow you to fine-tune the cost/accuracy trade-off. The following table shows all the available options:

| Option | Sub-option | Sub-sub-option | Description | Choices | Default |
|-|-|-|-|-|-|
| default_precision |  |  | The default precision to use for any PUB or `run()`<br/>call that does not specify one.<br/>Each input PUB can specify its own precision. If the `run()` method is given a precision, then that value is used for all PUBs in the `run()` call that do not specify their own.  | float > 0 | 0.015625 |
| max_execution_time |  |  | Maximum execution time in seconds, which is based<br/>on QPU usage (not wall clock time). QPU usage is the<br/>amount of time that the QPU is dedicated to processing your job. If a job exceeds this time limit, it is forcibly canceled. | Integer number of seconds in the range [1, 10800] |  |
| mitigation_level |  |  | How much error suppression and mitigation to apply. Refer to the [Mitigation level](#mitigation-level) section for more information about the methods used at each level. | 1 / 2 / 3 | 1 |
| optimization_level |  |  | How much optimization to perform on the circuits. [Higher levels](/docs/guides/set-optimization) generate more optimized circuits, at the expense of longer transpilation time. | 1 / 2 / 3 | 2 |
| dynamical_decoupling | enable |  | Whether to enable dynamical decoupling. Refer to [Error suppression and mitigation techniques](/docs/guides/error-mitigation-and-suppression-techniques#dynamical-decoupling) for the explanation of the method.  | True/False | True |
|  | sequence_type |  | Which dynamical decoupling sequence to use.<br/>* `XX`: use the sequence `tau/2 - (+X) - tau - (+X) - tau/2`<br/>* `XpXm`: use the sequence `tau/2 - (+X) - tau - (-X) - tau/2`<br/>* `XY4`: use the sequence<br/>`tau/2 - (+X) - tau - (+Y) - tau (-X) - tau - (-Y) - tau/2` | 'XX'/'XpXm'/'XY4' | 'XX' |
| twirling | enable_gates |  | Whether to apply 2-qubit Clifford gate twirling. | True/False | False |
|  | enable_measure |  | Whether to enable twirling of measurements. | True/False | True |
| resilience | measure_mitigation |  | Whether to enable TREX measurement error mitigation method. Refer to [Error suppression and mitigation techniques](/docs/guides/error-mitigation-and-suppression-techniques#twirled-readout-error-extinction-trex) for the explanation of the method.  | True/False | True |
|  | zne_mitigation |  | Whether to turn on Zero Noise Extrapolation error mitigation method. Refer to [Error suppression and mitigation techniques](/docs/guides/error-mitigation-and-suppression-techniques#zero-noise-extrapolation-zne) for the explanation of the method.  | True/False | False |
|  | zne | amplifier | Which technique to use for amplifying noise. One of: <br/> - `gate_folding` (default) uses 2-qubit gate folding to amplify noise. If the noise factor requires amplifying only a subset of the gates, then these gates are chosen randomly.<br/><br/> - `gate_folding_front` uses 2-qubit gate folding to amplify noise. If the noise factor requires amplifying only a subset of the gates, then these gates are selected from the front of the topologically ordered DAG circuit.<br/><br/> - `gate_folding_back` uses 2-qubit gate folding to amplify noise. If the noise factor requires amplifying only a subset of the gates, then these gates are selected from the back of the topologically ordered DAG circuit.<br/><br/> - `pea` uses a technique called Probabilistic error amplification (PEA) to amplify noise. Refer to [Error suppression and mitigation techniques](/docs/guides/error-mitigation-and-suppression-techniques#probabilistic-error-amplification-pea) for the explanation of the method.  | gate_folding / gate_folding_front / gate_folding_back / pea | gate_folding |
|  |  | noise_factors | Noise factors to use for noise amplification. | list of floats; each float >= 1 | (1, 1.5, 2) for PEA, and (1, 3, 5) otherwise. |
|  |  | extrapolator | Noise factors to evaluate the fit extrapolation models at. This option does not affect execution or model fitting in any way; it only determines the points at which the `extrapolator` objects are evaluated to be returned in the data fields called `evs_extrapolated` and `stds_extrapolated`. | one or more of `exponential`,`linear`, `double_exponential`,`polynomial_degree_(1 <= k <= 7)` | (`exponential`, `linear`) |
|  | pec_mitigation |  | Whether to turn on Probabilistic Error Cancellation error mitigation method. Refer to [Error suppression and mitigation techniques](/docs/guides/error-mitigation-and-suppression-techniques#probabilistic-error-cancellation-pec) for the explanation of the method.  | True/False | False |
|  | pec | max_overhead | The maximum circuit sampling overhead allowed, or `None` for no maximum. | None/ integer >1 | 100 |

In the following example, setting the mitigation level to 1 initially turns off ZNE mitigation, but setting `zne_mitigation` to `True` overrides the relevant setup from `mitigation_level`.

In [6]:
options = {"mitigation_level": 1, "resilience": {"zne_mitigation": True}}

## Input
Consulta la tabella seguente per tutti i parametri di input accettati da IBM Circuit function. La successiva sezione [Opzioni](#options) fornisce ulteriori dettagli sulle `options` disponibili.

| Nome      | Tipo                       | Descrizione                                                                                                                                                                                                                         | Obbligatorio | Default                                                                  | Esempio                                  |
|-----------|----------------------------|-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------|----------|--------------------------------------------------------------------------|------------------------------------------|
| backend_name   | str                        | Nome del backend su cui eseguire la query.                                                                                                                                                                                              | Sì      | N/A                                                                      | `ibm_fez`                                |
| pubs      | Iterable[EstimatorPubLike] | Un iterabile di oggetti PUB-like astratti (primitive unified bloc), come tuple `(circuit, observables)` o `(circuit, observables, parameter_values)`. Consulta [Panoramica dei PUB](/guides/primitive-input-output#overview-of-pubs) per ulteriori informazioni. I circuiti possono essere astratti (non-ISA). | Sì      | N/A                                                                      | (circuit, observables, parameter_values) |
| options   | dict                       | Opzioni di input. Consulta la sezione **Opzioni** per maggiori dettagli.                                                                                                                                                                                | No       | Consulta la sezione **Opzioni** per i dettagli.                                                   | `{"optimization_level": 3}`                |
| instance  | str                        | Il cloud resource name dell'istanza da utilizzare in quel formato.                                                                                                                                                                                        | No       | Ne viene scelta una casualmente se il tuo account ha accesso a più istanze. | `CRN`                   |

### Opzioni
#### Struttura
Analogamente alle primitive Qiskit Runtime, le opzioni per IBM Circuit function possono essere specificate come dizionario annidato. Le opzioni più comuni, come ``optimization_level`` e ``mitigation_level``, si trovano al primo livello. Altre opzioni più avanzate sono raggruppate in diverse categorie, come ``resilience``.

#### Valori predefiniti
Se non specifichi un valore per un'opzione, viene utilizzato il valore predefinito specificato dal servizio.

#### Livello di mitigazione
IBM Circuit function supporta anche `mitigation_level`. Il livello di mitigazione specifica quanta soppressione e mitigazione degli errori applicare al job. Livelli più alti producono risultati più accurati, a scapito di tempi di elaborazione più lunghi. Il grado di riduzione degli errori dipende dal metodo applicato. Il livello di mitigazione astrae la scelta dettagliata dei metodi di mitigazione e soppressione degli errori per consentire agli utenti di ragionare sul compromesso costo/accuratezza appropriato per la loro applicazione. La tabella seguente mostra i metodi corrispondenti per ciascun livello.

> **Note:** Sebbene i nomi siano simili, `mitigation_level` applica tecniche diverse da quelle usate da `resilience_level` dell'Estimator.

Analogamente a ``resilience_level`` in Qiskit Runtime Estimator, ``mitigation_level`` specifica un insieme base di opzioni curate. Tutte le opzioni che specifichi manualmente in aggiunta al livello di mitigazione vengono applicate sopra l'insieme base di opzioni definito dal livello di mitigazione. Quindi, in linea di principio, potresti impostare il livello di mitigazione a 1 e poi disattivare la mitigazione delle misure, anche se ciò non è consigliato.

| **Livello di mitigazione** | **Tecnica** |
|:-:|:-:|
| 1 [Predefinito] | Dynamical decoupling + measurement twirling + TREX  |
| 2 | Livello 1 + gate twirling + ZNE tramite gate folding |
| 3 | Livello 1 + gate twirling + ZNE tramite PEA |

L'esempio seguente mostra come impostare il livello di mitigazione:

In [7]:
print(f"The result of the submitted job had {len(result)} PUB")
print(
    f"The expectation values measured from this PUB are: \n{result[0].data.evs}"
)
print(f"And the associated metadata is: \n{result[0].metadata}")

The result of the submitted job had 1 PUB
The expectation values measured from this PUB are: 
1.02116704805492
And the associated metadata is: 
{'shots': 4096, 'target_precision': 0.015625, 'circuit_metadata': {}, 'resilience': {}, 'num_randomizations': 32}


#### Tutte le opzioni disponibili
Oltre a `mitigation_level`, IBM Circuit function offre anche un numero selezionato di opzioni avanzate che ti permettono di affinare il compromesso costo/accuratezza. La tabella seguente mostra tutte le opzioni disponibili:

| Opzione | Sotto-opzione | Sotto-sotto-opzione | Descrizione | Valori | Default |
|-|-|-|-|-|-|
| default_precision |  |  | La precisione predefinita da usare per qualsiasi PUB o chiamata `run()`<br/>che non ne specifichi una.<br/>Ogni PUB di input può specificare la propria precisione. Se al metodo `run()` viene fornita una precisione, tale valore viene usato per tutti i PUB nella chiamata `run()` che non specificano la propria.  | float > 0 | 0.015625 |
| max_execution_time |  |  | Tempo massimo di esecuzione in secondi, basato<br/>sull'utilizzo della QPU (non sul tempo reale). L'utilizzo della QPU è il<br/>tempo in cui la QPU è dedicata all'elaborazione del tuo job. Se un job supera questo limite di tempo, viene annullato forzatamente. | Numero intero di secondi nell'intervallo [1, 10800] |  |
| mitigation_level |  |  | Quanta soppressione e mitigazione degli errori applicare. Consulta la sezione [Livello di mitigazione](#mitigation-level) per ulteriori informazioni sui metodi usati a ciascun livello. | 1 / 2 / 3 | 1 |
| optimization_level |  |  | Quanta ottimizzazione eseguire sui circuiti. I [livelli più alti](/guides/set-optimization) generano circuiti più ottimizzati, a scapito di tempi di transpilazione più lunghi. | 1 / 2 / 3 | 2 |
| dynamical_decoupling | enable |  | Se abilitare il dynamical decoupling. Consulta [Tecniche di soppressione e mitigazione degli errori](/guides/error-mitigation-and-suppression-techniques#dynamical-decoupling) per la spiegazione del metodo.  | True/False | True |
|  | sequence_type |  | Quale sequenza di dynamical decoupling usare.<br/>* `XX`: usa la sequenza `tau/2 - (+X) - tau - (+X) - tau/2`<br/>* `XpXm`: usa la sequenza `tau/2 - (+X) - tau - (-X) - tau/2`<br/>* `XY4`: usa la sequenza<br/>`tau/2 - (+X) - tau - (+Y) - tau (-X) - tau - (-Y) - tau/2` | 'XX'/'XpXm'/'XY4' | 'XX' |
| twirling | enable_gates |  | Se applicare il gate twirling su gate Clifford a 2 qubit. | True/False | False |
|  | enable_measure |  | Se abilitare il twirling delle misure. | True/False | True |
| resilience | measure_mitigation |  | Se abilitare il metodo di mitigazione degli errori di misura TREX. Consulta [Tecniche di soppressione e mitigazione degli errori](/guides/error-mitigation-and-suppression-techniques#twirled-readout-error-extinction-trex) per la spiegazione del metodo.  | True/False | True |
|  | zne_mitigation |  | Se attivare il metodo di mitigazione degli errori Zero Noise Extrapolation. Consulta [Tecniche di soppressione e mitigazione degli errori](/guides/error-mitigation-and-suppression-techniques#zero-noise-extrapolation-zne) per la spiegazione del metodo.  | True/False | False |
|  | zne | amplifier | Quale tecnica usare per amplificare il rumore. Una tra: <br/> - `gate_folding` (default) usa il gate folding a 2 qubit per amplificare il rumore. Se il fattore di rumore richiede di amplificare solo un sottoinsieme di gate, questi vengono scelti casualmente.<br/><br/> - `gate_folding_front` usa il gate folding a 2 qubit per amplificare il rumore. Se il fattore di rumore richiede di amplificare solo un sottoinsieme di gate, questi vengono selezionati dall'inizio del circuito DAG ordinato topologicamente.<br/><br/> - `gate_folding_back` usa il gate folding a 2 qubit per amplificare il rumore. Se il fattore di rumore richiede di amplificare solo un sottoinsieme di gate, questi vengono selezionati dalla fine del circuito DAG ordinato topologicamente.<br/><br/> - `pea` usa una tecnica chiamata Probabilistic Error Amplification (PEA) per amplificare il rumore. Consulta [Tecniche di soppressione e mitigazione degli errori](/guides/error-mitigation-and-suppression-techniques#probabilistic-error-amplification-pea) per la spiegazione del metodo.  | gate_folding / gate_folding_front / gate_folding_back / pea | gate_folding |
|  |  | noise_factors | Fattori di rumore da usare per l'amplificazione del rumore. | lista di float; ogni float >= 1 | (1, 1.5, 2) per PEA, e (1, 3, 5) negli altri casi. |
|  |  | extrapolator | Fattori di rumore a cui valutare i modelli di estrapolazione del fit. Questa opzione non influisce sull'esecuzione né sul fitting del modello in alcun modo; determina solo i punti in cui gli oggetti `extrapolator` vengono valutati per essere restituiti nei campi dati chiamati `evs_extrapolated` e `stds_extrapolated`. | uno o più tra `exponential`,`linear`, `double_exponential`,`polynomial_degree_(1 <= k <= 7)` | (`exponential`, `linear`) |
|  | pec_mitigation |  | Se attivare il metodo di mitigazione degli errori Probabilistic Error Cancellation. Consulta [Tecniche di soppressione e mitigazione degli errori](/guides/error-mitigation-and-suppression-techniques#probabilistic-error-cancellation-pec) per la spiegazione del metodo.  | True/False | False |
|  | pec | max_overhead | Il massimo overhead di campionamento del circuito consentito, o `None` per nessun massimo. | None/ intero >1 | 100 |

Nell'esempio seguente, impostare il livello di mitigazione a 1 disattiva inizialmente la mitigazione ZNE, ma impostare `zne_mitigation` a `True` sovrascrive la configurazione corrispondente di `mitigation_level`.

In [8]:
job = function.run(
    backend_name="bad_backend_name", pubs=pubs, options=options
)

print(job.result())

QiskitServerlessException: "Traceback (most recent call last):\n  File \"/runner/runner.py\", line 10, in run\n    func = CircuitFunction(**arguments)\n           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^\n  File \"/runner/circuit_function/circuit_function.py\", line 87, in __init__\n    self._backend = self._service.backend(\n                    ^^^^^^^^^^^^^^^^^^^^^^\n  File \"/usr/local/lib/python3.11/site-packages/qiskit_ibm_runtime/qiskit_runtime_service.py\", line 754, in backend\n    backends = self.backends(name, instance=instance, use_fractional_gates=use_fractional_gates)\n               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^\n  File \"/usr/local/lib/python3.11/site-packages/qiskit_ibm_runtime/qiskit_runtime_service.py\", line 497, in backends\n    raise QiskitBackendNotFoundError(\"No backend matches the criteria.\")\nqiskit.providers.exceptions.QiskitBackendNotFoundError: 'No backend matches the criteria.'\n"

## Output
L'output di una Circuit function è un [PrimitiveResult](https://docs.quantum.ibm.com/api/qiskit/qiskit.primitives.PrimitiveResult), che contiene due campi:

- Uno o più oggetti [PubResult](https://docs.quantum.ibm.com/api/qiskit/qiskit.primitives.PubResult). Questi possono essere indicizzati direttamente dal `PrimitiveResult`.

- Metadati a livello di job.

Ogni `PubResult` contiene un campo `data` e un campo `metadata`.

- Il campo `data` contiene almeno un array di valori di aspettazione (`PubResult.data.evs`) e un array di errori standard (`PubResult.data.stds`). Può contenere anche altri dati, a seconda delle opzioni usate.

- Il campo `metadata` contiene i metadati a livello di PUB (`PubResult.metadata`).

Il seguente snippet di codice descrive il formato di `PrimitiveResult` (e del relativo `PubResult`).